In [ ]:
from typing import Annotated , Literal
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import tool
from langchain_experimental.utilities import PythonREPL
from typing_extensions import TypedDict
from langgraph.graph import START , END , MessagesState, StateGraph
from langgraph.types import Command
from langchain.messages import HumanMessage
from langchain.agents import create_agent

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite",)

In [ ]:
search_engine = TavilySearchResults(
    max_results=5,
    topic="general",
    search_depth="basic",
    time_range="week",
)

In [ ]:
repl = PythonREPL()

In [ ]:
code = """
def add(a, b):
    return a + b
result = add(2, 3)
print(result)
"""

In [ ]:
repl.run(code)

In [ ]:
# @tool
def python_repl_tool(
    code: Annotated[str, "The python code to execute to generate your chart."],
):
    """Use this to execute python code and do math. If you want to see the output of a value,
    you should print it out with `print(...)`. This is visible to the user."""

    try:
        result = repl.run(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"

    result_str = f"Successfully executed:\n```python\n{code}\n```\nStdout: {result}"
    return result_str

In [ ]:
print(python_repl_tool(code))

In [ ]:
members = ['researcher','coder','FINISH']

In [ ]:
members

In [ ]:
class Router(TypedDict):
    """Worker router to determine which agent to send the message to next.If no more agents are needed, route to 'FINISH'."""
    next: Literal['researcher','coder','FINISH'] 

In [ ]:
class State(MessagesState):
    next : str

In [ ]:
system_prompt = f"""
You are a supervisor managing these workers:
{members}
- researcher: gathers information
- coder: performs calculations using Python

Rules:

1. If factual information must be gathered,
   route to researcher.

2. If any calculation, projection,
   percentage, growth rate, formula,
   statistics, or numerical analysis is needed,
   route to coder.

3. If researcher returns
   CALCULATION_REQUIRED,
   always route to coder.

4. Only return FINISH when both
   information gathering and calculations
   are complete.
"""

In [ ]:
[{"role": "system", "content": system_prompt},]

In [ ]:
state={"next":["hi"]}

In [ ]:
state["next"]

In [ ]:
[{"role": "system", "content": system_prompt},] + state["next"]

In [ ]:
def supervisor_node(state: State) -> Command[Literal["researcher", "coder", "__end__"]]:
    
    messages = [{"role": "system", "content": system_prompt},] + state["messages"]
    
    response = model.with_structured_output(Router).invoke(messages)
    
    goto = response["next"]
    
    print("below my goto**********************************")
    
    print(goto)
    
    if goto == "FINISH":
        goto = END
        
    return Command(goto=goto, update={"next": goto})

In [ ]:
researcher_prompt ="""
You are a researcher.

Your ONLY responsibility is gathering facts.

You may use Tavily.

Never:
- perform calculations
- estimate values
- write Python code
- apply formulas
- provide projected results

If calculations are required,
return only the raw information needed.

End your response with:
CALCULATION_REQUIRED
"""

In [ ]:
def research_node(state: State) -> Command[Literal["supervisor"]]:
    
    research_agent = create_agent(model, tools=[search_engine], system_prompt=researcher_prompt)
    
    result = research_agent.invoke(state)
    
    return Command(
        update={
            "messages": [
                HumanMessage(content=result["messages"][-1].content, name="researcher")
            ]
        },
        goto="supervisor",
    )

In [ ]:
def code_node(state: State) -> Command[Literal["supervisor"]]:
    
    code_agent = create_agent(model, tools=[python_repl_tool])
    
    result = code_agent.invoke(state)
    
    return Command(
        update={
            "messages": [
                HumanMessage(content=result["messages"][-1].content, name="coder")
            ]
        },
        goto="supervisor",
    )

In [ ]:
graph = StateGraph(State)

In [ ]:
graph.add_node("supervisor", supervisor_node)
graph.add_node("researcher", research_node)
graph.add_node("coder", code_node)

In [ ]:
graph.add_edge(START, "supervisor")


In [ ]:
app = graph.compile()

In [ ]:
app

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage


def extract_content(msg):
    content = msg.content

    # Gemini content blocks
    if isinstance(content, list):
        texts = []

        for block in content:
            if isinstance(block, dict):
                if block.get("type") == "text":
                    texts.append(block.get("text", ""))

        return "\n".join(texts)

    return str(content)

In [ ]:
from rich.console import Console
from rich.panel import Panel

console = Console()

for path, data in app.stream(
    {"messages": [("user", "Find the latest population of Canada from a reliable source, then calculate its projected population after 15 years assuming 0.8% annual compound growth. Show the formula, Python calculation, and final answer.")]},
    subgraphs=True,
):

    for node_name, node_output in data.items():

        if "next" in node_output:
            console.print(
                Panel(
                    node_output["next"],
                    title=f"{node_name} → Next",
                    border_style="green",
                )
            )

        if "messages" in node_output:
            for msg in node_output["messages"]:

                if isinstance(msg, AIMessage):
                    msg_type = "AI"

                elif isinstance(msg, HumanMessage):
                    msg_type = "Human"

                elif isinstance(msg, ToolMessage):
                    msg_type = "Tool"

                else:
                    msg_type = type(msg).__name__

                console.print(
                    Panel(
                        extract_content(msg),
                        title=f"{node_name} | {msg_type}",
                        border_style="cyan",
                    )
                )